# Part 2: Recompute the statisticsThis notebook builds directly on **Part 1**. Part 1 has already loaded the samples, parsed the recording markers, and created the record ranges. Here we reuse those objects and compute the statistics; we do not reload files or repeat marker parsing.The formulas below are inferred from the task geometry and checked against the reference values. We keep full precision during computation and round only for display.

## 1. Definitions usedFor every recorded sample, let `r = sqrt((mouseX - centerX)^2 + (mouseY - centerY)^2)`. The angle is calculated around the task centre after reversing the screen Y axis, then unwrapped so rotations crossing the angle boundary remain continuous.- **nLaps** = total signed unwrapped angular displacement / `2π`. Negative values indicate the direction seen in the provided reference data.- **Re** = mean sample radius `r` (pixels).- **Te** = `4.133 × SD(r)` (pixels), the conventional effective width based on radial variability.- **error** = 100 × samples with `mouseInTarget == 0` / all samples (%).- **MT/lap** = elapsed sample time / `abs(nLaps)` (s/lap).- **IDe/lap** = `2π × Re / Te` (bit/lap as used by the reference table).- **Be** = `Te / taskTolerance` (double).- **IPe** = `(IDe/lap) / (MT/lap)` (bit/s).The six-cycle setup has only five completed `DoRecord` intervals in the supplied file. Therefore `Rec006` is not computed from samples; the zero/NaN row in the marker table is a placeholder for the uncompleted sixth interval.**Choice to verify if your output differs slightly:** this notebook uses population SD (`np.std`, `ddof=0`) and the first-to-last sample duration. Compare the printed table with the marker table; if `Te` differs by a small amount, try sample SD (`ddof=1`) or marker-to-marker duration for `MT/lap` and note which matches better.

## 2. Reuse Part 1 data and task settings

In [ ]:
import numpy as np# Part 1 creates these variables and helper functions.required = ["time_ms", "x", "y", "inside", "record_ranges", "starts", "extract_record", "CENTER_X", "CENTER_Y", "TASK_RADIUS", "TASK_TOL"]missing = [name for name in required if name not in globals()]if missing:    raise RuntimeError("Run Part 1 before Part 2 (main.ipynb does this automatically). Missing: " + ", ".join(missing))print("Reusing Part 1 data and settings.")print("Completed recording intervals:", len(record_ranges))for i, (i0, i1) in enumerate(record_ranges, 1):    print(f"Rec{i:03d}: {i1-i0} samples")

## 3. Compute the statistics for each completed recording

In [ ]:
def compute_record_statistics(i0, i1, ddof=0):    """Compute one statistics row using Part 1's record extraction helper."""    time_s, x_rel, y_rel, inside_rec = extract_record(time_ms, x, y, inside, i0, i1, starts[0])    r = np.hypot(x_rel, y_rel)    theta = np.unwrap(np.arctan2(y_rel, x_rel))    n_laps = (theta[-1] - theta[0]) / (2 * np.pi)    re = np.mean(r)    te = 4.133 * np.std(r, ddof=ddof) if len(r) > ddof else np.nan    err = 100 * np.mean(inside_rec == 0)    duration_s = time_s[-1] - time_s[0]    mt_per_lap = duration_s / abs(n_laps) if n_laps != 0 else np.nan    ide_per_lap = (2 * np.pi * re / te) if te > 0 else np.nan    be = te / TASK_TOL if np.isfinite(te) else np.nan    ipe = ide_per_lap / mt_per_lap if mt_per_lap > 0 else np.nan    return {"nLaps": n_laps, "Re": re, "Te": te, "error": err,            "MT/lap": mt_per_lap, "IDe/lap": ide_per_lap, "Be": be, "IPe": ipe,            "samples": len(r), "duration_s": duration_s}results = [compute_record_statistics(i0, i1) for i0, i1 in record_ranges]for i, row in enumerate(results, 1):    print(f"Rec{i:03d}: nLaps={row['nLaps']:.4f}, Re={row['Re']:.3f}, Te={row['Te']:.3f}, "          f"error={row['error']:.3f}%, MT/lap={row['MT/lap']:.4f}s, "          f"IDe/lap={row['IDe/lap']:.3f}, Be={row['Be']:.3f}, IPe={row['IPe']:.3f}")

## 4. Display and compare with the reference table

In [ ]:
columns = ["Var", "nLaps", "Re", "Te", "error", "MT/lap", "IDe/lap", "Be", "IPe"]print(" | ".join(f"{name:>9}" for name in columns))print("-" * 105)print(" | ".join([f"{'Theory':>9}", f"{1:9.2f}", f"{TASK_RADIUS:9.2f}", f"{TASK_TOL:9.2f}",                 f"{3.88:9.2f}", f"{':>9}", f"{':>9}", f"{1:9.2f}", f"{':>9}"]))for i, row in enumerate(results, 1):    vals = [f"Rec{i:03d}", f"{row['nLaps']:.2f}", f"{row['Re']:.2f}", f"{row['Te']:.2f}",            f"{row['error']:.2f}", f"{row['MT/lap']:.2f}", f"{row['IDe/lap']:.2f}",            f"{row['Be']:.2f}", f"{row['IPe']:.2f}"]    print(" | ".join(f"{v:>9}" for v in vals))if len(results) < 6:    print(" | ".join([f"{'Rec006':>9}", *[f"{v:>9}" for v in ["0.00", "0.00", "0.00", "NaN", "NaN", "NaN", "0.00", "0.00"]]))print("\nReference values are embedded below for comparison (rounded as in the assignment).")reference = np.array([    [-10.84, 213.86, 91.78, 3.90, 1.84, 14.64, 1.95, 7.94],    [-11.97, 209.85, 34.45, 0.74, 1.59, 38.27, 0.73, 24.04],    [-13.26, 209.43, 42.42, 2.81, 1.45, 31.02, 0.90, 21.40],    [-12.83, 209.46, 37.14, 0.13, 1.49, 35.44, 0.79, 23.72],    [-14.74, 210.89, 43.88, 2.84, 1.30, 30.20, 0.93, 23.20],])print("Column order: nLaps, Re, Te, error, MT/lap, IDe/lap, Be, IPe")for i, row in enumerate(results[:len(reference)], 1):    calculated = np.array([row[k] for k in ["nLaps", "Re", "Te", "error", "MT/lap", "IDe/lap", "Be", "IPe"]])    print(f"Rec{i:03d} calculated - reference:", np.round(calculated - reference[i-1], 3))